In [1]:
import pandas as pd
import numpy as np

In [3]:
temp_df = pd.read_csv('IMDB Dataset.csv')

In [5]:
df = temp_df[:10000]

In [7]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [9]:
df.drop_duplicates(inplace=True)

C:\Users\DELL\AppData\Local\Temp\ipykernel_19064\3006716147.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace=True)


In [11]:
import re
def remove_tags(raw_text):
    cleaned_text = re.sub(re.compile('<.*?>'), '', raw_text)
    return cleaned_text

In [13]:
df['review'] = df['review'].apply(remove_tags)

C:\Users\DELL\AppData\Local\Temp\ipykernel_19064\2336150696.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(remove_tags)


In [15]:
df['review'] = df['review'].apply(lambda x: x.lower())

C:\Users\DELL\AppData\Local\Temp\ipykernel_19064\2432328559.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(lambda x: x.lower())


In [17]:
from nltk.corpus import stopwords

sw_list = stopwords.words('english')

df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))

C:\Users\DELL\AppData\Local\Temp\ipykernel_19064\2826946130.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))


In [18]:
df['review']

0       one reviewers mentioned watching 1 oz episode ...
1       wonderful little production. filming technique...
2       thought wonderful way spend time hot summer we...
3       basically there's family little boy (jake) thi...
4       petter mattei's "love time money" visually stu...
                              ...                        
9995    fun, entertaining movie wwii german spy (julie...
9996    give break. anyone say "good hockey movie"? kn...
9997    movie bad movie. watching endless series bad h...
9998    movie probably made entertain middle school, e...
9999    smashing film film-making. shows intense stran...
Name: review, Length: 9983, dtype: object

In [21]:
import gensim

In [23]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [25]:
story = []
for doc in df['review']:
    raw_sent = sent_tokenize(doc)
    for sent in raw_sent:
        story.append(simple_preprocess(sent))

In [27]:
model = gensim.models.Word2Vec(
    window=10,
    min_count=2
)

In [29]:
model.build_vocab(story)

In [31]:
model.train(story, total_examples=model.corpus_count, epochs=model.epochs)

(5850302, 6186875)

In [33]:
len(model.wv.index_to_key)

31845

In [35]:
def document_vector(doc):
    #remove out of vocabulary words
    doc = [word for word in doc.split() if word in model.wv.index_to_key]
    return np.mean(model.wv[doc], axis=0)

In [37]:
document_vector(df['review'].values[0])

array([ 5.14096953e-02,  3.76580834e-01,  1.91871732e-01,  3.88737448e-04,
       -2.98310407e-02, -5.40555060e-01,  1.21615849e-01,  7.79465735e-01,
       -2.18840390e-01, -2.08294332e-01, -2.39006236e-01, -5.18601656e-01,
       -2.78806966e-02,  3.03443909e-01,  2.25099444e-01, -2.26800859e-01,
        1.75612539e-01, -5.61712027e-01, -4.47314531e-02, -6.58186018e-01,
        1.85640454e-02,  1.01057693e-01,  1.78102314e-01, -3.22121590e-01,
       -2.28857070e-01, -1.99307948e-01, -1.31242171e-01, -1.93821609e-01,
       -4.33762997e-01, -1.12038784e-01,  4.15172815e-01, -1.00317691e-02,
       -5.78544512e-02, -2.28185847e-01, -3.89415890e-01,  3.11631650e-01,
        1.54330991e-02, -2.41131485e-01, -1.67336851e-01, -7.59694815e-01,
        1.44315124e-01, -1.98069811e-01, -1.13032728e-01, -2.05097292e-02,
        4.25586700e-01, -1.72407925e-01, -4.24300849e-01, -2.02802643e-01,
        4.25218083e-02,  2.91593343e-01,  1.85746774e-01, -2.86517918e-01,
       -3.54746491e-01, -

In [39]:
from tqdm import tqdm

In [43]:
X = []
for doc in tqdm(df['review'].values):
    X.append(document_vector(doc))

100%|██████████████████████████████████████████████████████████████████████████████| 9983/9983 [23:16<00:00,  7.15it/s]


In [45]:
X = np.array(X)

In [47]:
X[0]

array([ 5.14096953e-02,  3.76580834e-01,  1.91871732e-01,  3.88737448e-04,
       -2.98310407e-02, -5.40555060e-01,  1.21615849e-01,  7.79465735e-01,
       -2.18840390e-01, -2.08294332e-01, -2.39006236e-01, -5.18601656e-01,
       -2.78806966e-02,  3.03443909e-01,  2.25099444e-01, -2.26800859e-01,
        1.75612539e-01, -5.61712027e-01, -4.47314531e-02, -6.58186018e-01,
        1.85640454e-02,  1.01057693e-01,  1.78102314e-01, -3.22121590e-01,
       -2.28857070e-01, -1.99307948e-01, -1.31242171e-01, -1.93821609e-01,
       -4.33762997e-01, -1.12038784e-01,  4.15172815e-01, -1.00317691e-02,
       -5.78544512e-02, -2.28185847e-01, -3.89415890e-01,  3.11631650e-01,
        1.54330991e-02, -2.41131485e-01, -1.67336851e-01, -7.59694815e-01,
        1.44315124e-01, -1.98069811e-01, -1.13032728e-01, -2.05097292e-02,
        4.25586700e-01, -1.72407925e-01, -4.24300849e-01, -2.02802643e-01,
        4.25218083e-02,  2.91593343e-01,  1.85746774e-01, -2.86517918e-01,
       -3.54746491e-01, -

In [49]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

y = encoder.fit_transform(df['sentiment'])

In [51]:
y

array([1, 1, 1, ..., 0, 0, 1])

In [53]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.2, random_state=1)

In [55]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [57]:
rf = RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test,y_pred)

0.7751627441161743